In [ ]:
# Connects the Colab environment to Drive environment.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**DATASET:** This first section consists of installing neccesary libraries/packages for downloading and retrieve the dataset from kaggle.

In [ ]:
!pip install -U kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.8/217.8 kB 18.0 MB/s eta 0:00:00
  Attempting uninstall: kagglesdk
    Found existing installation: kagglesdk 0.1.20
    Uninstalling kagglesdk-0.1.20:
      Successfully uninstalled kagglesdk-0.1.20
  Attempting uninstall: kaggle
    Found existing installation: kaggle 2.0.2
    Uninstalling kaggle-2.0.2:
      Successfully uninstalled kaggle-2.0.2


In [ ]:
# Upload kaggle.json file to be able to get the desired kaggle dataset.
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"dv22fae","key":"362422990969946184f6a62e0ddf2b18"}'}

In [ ]:
!mkdir ~/.kaggle

In [ ]:
!cp kaggle.json ~/.kaggle/

In [ ]:
!ls -ltr ~/.kaggle

total 4
-rw-r--r-- 1 root root 63 May 23 09:34 kaggle.json


In [ ]:
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# List the desired dataset
!kaggle datasets list -s 'Data_Set_Spruce_Bark_Beetle'

ref                                 title                              size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------  ---------------------------  ----------  --------------------------  -------------  ---------  ---------------  
tanchu/data-set-spruce-bark-beetle  Data_Set_Spruce_Bark_Beetle  6876892793  2023-06-10 12:00:45.557000            379         13  0.6875           


In [ ]:
from pathlib import Path
google_drive_path = Path("/content/drive/MyDrive/kaggle_data_set")
zip_file_path = google_drive_path / "data-set-spruce-bark-beetle.zip"
local_destination_path = Path("/content/data")

# Creates the map structure if it does not already exist
google_drive_path.mkdir(parents=True, exist_ok=True)
local_destination_path.mkdir(parents=True, exist_ok=True)

# Download dataset to drive and then unzip to colab.
if not zip_file_path.exists():
  !kaggle datasets download -d 'tanchu/data-set-spruce-bark-beetle' -p {google_drive_path}
  !unzip {zip_file_path} -d {local_destination_path}

else:
  !unzip {zip_file_path} -d {local_destination_path}


Utdata för streaming har trunkerats till de sista 5000 raderna.
  inflating: /content/data/Data_Set_Spruce_Bark_Beetle/oblique/Backsjon_20201016_oblique/Images/27635e6e-b4d9-4517-8310-e0a517d055ca.jpg  
  inflating: /content/data/Data_Set_Spruce_Bark_Beetle/oblique/Backsjon_20201016_oblique/Images/27675ad7-f570-4da3-8f54-10fb4da20cea.jpg  
  inflating: /content/data/Data_Set_Spruce_Bark_Beetle/oblique/Backsjon_20201016_oblique/Images/27fc1531-788a-4a4a-8a09-622f08f923b3.jpg  
  inflating: /content/data/Data_Set_Spruce_Bark_Beetle/oblique/Backsjon_20201016_oblique/Images/285b2da9-ccbb-4bc2-9e5d-c328fd12719e.jpg  
  inflating: /content/data/Data_Set_Spruce_Bark_Beetle/oblique/Backsjon_20201016_oblique/Images/2900cb38-5f7f-4d14-8986-155e512a809a.jpg  
  inflating: /content/data/Data_Set_Spruce_Bark_Beetle/oblique/Backsjon_20201016_oblique/Images/2a6dfc67-0ba2-43a3-8e84-d9a1110cabec.jpg  
  inflating: /content/data/Data_Set_Spruce_Bark_Beetle/oblique/Backsjon_20201016_oblique/Images/2aaa53

**Setup Folder Structure**. This section consists of setting up the folder structure for cross validation and to suit YOLO models.

In [ ]:
num_splits = 4
from pathlib import Path
for i in range(1, num_splits + 1):
  Path(f"/content/data/folder_{i}/train/images").mkdir(parents=True, exist_ok=True)
  Path(f"/content/data/folder_{i}/train/labels").mkdir(parents=True, exist_ok=True)

  Path(f"/content/data/folder_{i}/val/images").mkdir(parents=True, exist_ok=True)
  Path(f"/content/data/folder_{i}/val/labels").mkdir(parents=True, exist_ok=True)

  Path(f"/content/data/folder_{i}/images").mkdir(parents=True, exist_ok=True)
  Path(f"/content/data/folder_{i}/labels").mkdir(parents=True, exist_ok=True)

Path("/content/data/test/images").mkdir(parents=True, exist_ok=True)
Path("/content/data/test/labels").mkdir(parents=True, exist_ok=True)

**Data conversion** Convert data to YOLO format

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.0 MB/s eta 0:00:00


In [ ]:
import xml.etree.ElementTree as annotations_tree
import os
import pandas as pd
import shutil
from ultralytics.utils.ops import xyxy2xywhn
import torch
import random

# Paths to annottation folders
annotation_folder_backsjon = "/content/data/Data_Set_Spruce_Bark_Beetle/damage/Backsjon_20201012_damage/Annotations"
annotation_folder_Viken = "/content/data/Data_Set_Spruce_Bark_Beetle/damage/Viken_20180918_damage/Annotations"
annotation_folder_Lidhem_oblique_may = "/content/data/Data_Set_Spruce_Bark_Beetle/damage/Lidhem_20200527_oblique_damage/Annotations"
annotation_folder_Lidhem_oblique_june = "/content/data/Data_Set_Spruce_Bark_Beetle/damage/Lidhem_20200614_oblique_damage/Annotations"
annotation_folder_Lidhem = "/content/data/Data_Set_Spruce_Bark_Beetle/damage/Lidhem_20201001_damage/Annotations"

# Path to iamges folders
image_folder_backsjon = "/content/data/Data_Set_Spruce_Bark_Beetle/damage/Backsjon_20201012_damage/Images"
image_folder_Viken = "/content/data/Data_Set_Spruce_Bark_Beetle/damage/Viken_20180918_damage/Images"
image_folder_Lidhem_oblique_may = "/content/data/Data_Set_Spruce_Bark_Beetle/damage/Lidhem_20200527_oblique_damage/Images"
image_folder_Lidhem_oblique_june = "/content/data/Data_Set_Spruce_Bark_Beetle/damage/Lidhem_20200614_oblique_damage/Images"
image_folder_Lidhem = "/content/data/Data_Set_Spruce_Bark_Beetle/damage/Lidhem_20201001_damage/Images"

# Creates tuples with paths to matching annotation and image folders
folders = [
		 (annotation_folder_backsjon, image_folder_backsjon),
		 (annotation_folder_Viken, image_folder_Viken),
		 (annotation_folder_Lidhem_oblique_may, image_folder_Lidhem_oblique_may),
		 (annotation_folder_Lidhem_oblique_june, image_folder_Lidhem_oblique_june),
		 (annotation_folder_Lidhem, image_folder_Lidhem)
]

# Dictionary for high damage (HD)
treetype_to_integers = {
	"HD": 0
}

# Keeps track on the number of files (used for cross validation split)
file_counter = 0
file_counter_train = 0

for annotation_folder, image_folder in folders:
	xml_files = []
	for file in os.listdir(annotation_folder):

		# Ensure we look at xml files
		if file.endswith(".xml"):
			xml_files.append(file)

	# Sorts the xml files to ensure they are in the same order between runs
	xml_files = sorted(xml_files)

	# Shuffles the data to decrease bias for images taken near the same time that might look similar (random.seed for reproducibility)
	random.seed(77)
	random.shuffle(xml_files)

	for xml_file in xml_files:
		file_counter += 1

		yolo_format_data = []

		# The entire path for parsing the file.
		full_xml_path = os.path.join(annotation_folder, xml_file)

		# Parse the xml file to extract data from it.
		parsed_xml_file = annotations_tree.parse(full_xml_path)
		root = parsed_xml_file.getroot()

		# Get the file name
		file_name = root.find("filename").text
		file_name = os.path.splitext(file_name)[0]

		# Get image size
		img_size = root.find("size")
		img_width = int(img_size.find("width").text)
		img_height = int(img_size.find("height").text)

		# For each object in the xml file
		for obj in root.findall("object"):

			# Only damaged object is included since only one class is used ("HD")
			damage = obj.find("damage").text
			if damage == "HD":

				# Get the bounding box dimensions for each object
				bounding_box = obj.find("bndbox")
				x_min = int(bounding_box.find("xmin").text)
				y_min = int(bounding_box.find("ymin").text)
				x_max = int(bounding_box.find("xmax").text)
				y_max = int(bounding_box.find("ymax").text)

				# The torch representation of the boundingbox.
				torch_box = torch.tensor([[x_min, y_min, x_max, y_max]], dtype=torch.float32)

				# Convert the boundingboxes safely to YOLO format.
				yolo_bounding_box = xyxy2xywhn(torch_box, img_width, img_height)

				# Extract the normalized YOLO values.
				x_center, y_center, width, height = yolo_bounding_box[0].tolist()

				# Saves the data
				yolo_format_data.append([
					treetype_to_integers[damage],
					x_center,
					y_center,
					width,
					height
				])

		# Test data (12.5%)
		if file_counter % 8 == 0:

			# Creates a dataframe with the YOLO format data and transfer it to a txt file
			data_frame = pd.DataFrame(yolo_format_data)
			storage_folder = "/content/data/test/labels/" + file_name + ".txt"
			data_frame.to_csv(storage_folder, sep=" ", index=False, header=False)

			# Copy corresponding image
			shutil.copy2(os.path.join(image_folder, file_name + ".jpg"), os.path.join("/content/data/test/images/", file_name + ".jpg"))
			#continue

		# Cross validation data (87.5%) is split into 4 (num_splits)
		else:
			file_counter_train += 1
			folder_pos = (file_counter_train % num_splits) + 1

			# Creates a dataframe with the YOLO format data and transfer it to a txt file
			data_frame = pd.DataFrame(yolo_format_data)
			storage_folder = f"/content/data/folder_{folder_pos}/labels/" + file_name + ".txt"
			data_frame.to_csv(storage_folder, sep=" ", index=False, header=False)

			# Copy corresponding image
			shutil.copy2(os.path.join(image_folder, file_name + ".jpg"), os.path.join(f"/content/data/folder_{folder_pos}/images/", file_name + ".jpg"))


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# Divides into cross validation folders
from pathlib import Path
import shutil

# Sort the data before divinding into cross validation structure
sorted_images_labels = {}
for i in range (1,num_splits + 1):
  sorted_images_labels[i] = {
      "images": sorted(Path(f"/content/data/folder_{i}/images").glob("*.jpg")),
      "labels": sorted(Path(f"/content/data/folder_{i}/labels").glob("*.txt"))
  }

for i in range (1,num_splits + 1):
  for j in range (1,num_splits + 1):

    # Copy files to train images and train labels
    if i != j:
      for image, label in zip(sorted_images_labels[j]["images"], sorted_images_labels[j]["labels"]):
        shutil.copy2(image, f"/content/data/folder_{i}/train/images")
        shutil.copy2(label, f"/content/data/folder_{i}/train/labels")

    # Copy files to val images and val labels
    else:
      for image, label in zip(sorted_images_labels[j]["images"], sorted_images_labels[j]["labels"]):
        shutil.copy2(image, f"/content/data/folder_{i}/val/images")
        shutil.copy2(label, f"/content/data/folder_{i}/val/labels")

# Removes folder and content that is not used
for i in range (1,num_splits + 1):
  shutil.rmtree(f"/content/data/folder_{i}/images")
  shutil.rmtree(f"/content/data/folder_{i}/labels")

In [ ]:
# Prints out the number of files in each folder to see the result of cross validation split
for i in range (1, num_splits + 1):
  print(f"folder_{i}/train number of images:  ", len(list(Path(f"/content/data/folder_{i}/train/images").glob("*"))))
  print(f"folder_{i}/train number of labels:  ", len(list(Path(f"/content/data/folder_{i}/train/labels").glob("*"))))

  print(f"folder_{i}/val number of images:  ", len(list(Path(f"/content/data/folder_{i}/val/images").glob("*"))))
  print(f"folder_{i}/val number of labels:  ", len(list(Path(f"/content/data/folder_{i}/val/labels").glob("*"))))

print("test number of images ",  len(list(Path("/content/data/test/images").glob("*"))))
print("test number of labels ", len(list(Path("/content/data/test/labels").glob("*"))))

folder_1/train number of images:   218
folder_1/train number of labels:   218
folder_1/val number of images:   72
folder_1/val number of labels:   72
folder_2/train number of images:   217
folder_2/train number of labels:   217
folder_2/val number of images:   73
folder_2/val number of labels:   73
folder_3/train number of images:   217
folder_3/train number of labels:   217
folder_3/val number of images:   73
folder_3/val number of labels:   73
folder_4/train number of images:   218
folder_4/train number of labels:   218
folder_4/val number of images:   72
folder_4/val number of labels:   72
test number of images  41
test number of labels  41


In [ ]:
from pathlib import Path
Path(f"/content/data/YAMLS").mkdir(parents=True, exist_ok=True)

# Creates yaml files for each folder for cross valdiaiton
for i in range(1,num_splits + 1):
  # Corresponding yaml file for each folder
  yaml_text = f"""path: /content/data/folder_{i}
train: train/images
val: val/images

nc: 1

names: ["HD"]
"""
  Path(f"/content/data/YAMLS/folder_{i}.yaml").write_text(yaml_text)

84

Cross validation preparation

In [ ]:
# Import YOLO to use YOLO models
from ultralytics import YOLO


In [ ]:
hyperparameter_config = {
    "epochs": 70,
    "imgsz": 1024,
    "batch": 10,
    "seed": 77,
    "deterministic": True,
}

In [ ]:
!pip install albumentations ultralytics

In [ ]:
import albumentations as data_aug

# Strong CLAHE while rest is default.
data_augmentation = [
    data_aug.Blur(p=0.01, blur_limit=(3, 7)),
    data_aug.MedianBlur(p=0.01, blur_limit=(3, 7)),
    data_aug.ToGray(p=0.01, method='weighted_average', num_output_channels=3),
    data_aug.CLAHE(p=0.5, clip_limit=(4.0, 8.0), tile_grid_size=(16, 16)),
]


**Training, cross validation and evaluation**

In [ ]:
%%time

# Get the correct YAML files
folder_yamls = []
for p in Path("/content/data/YAMLS").glob("folder_*.yaml"):
  folder_yamls.append(str(p))
folder_yamls.sort()

# Run cross validation and store performance metrics
cross_validation_results = []
for yaml in folder_yamls:
  model = YOLO("yolo26s.pt")
  model.train(data=str(yaml), **hyperparameter_config, augmentations=data_augmentation)
  result = model.val(data=str(yaml), classes=[0], plots=True, save_json=True)
  cross_validation_results.append(result)

# Calculate avergage metrics from cross validation
precision_damage = []
recall_damage = []
map_50 = []
map_50_95 = []
f1_score = []
time_metrics = []
for result in cross_validation_results:
  precision_damage.append(result.box.p[0]) # Index 0 represnts "HD" (high damage)
  recall_damage.append(result.box.r[0]) # Index 0 represnts "HD" (high damage)
  map_50.append(result.box.ap50[0]) # Index 0 represnts "HD" (high damage)
  map_50_95.append(result.box.maps[0]) # Index 0 represnts "HD" (high damage)
  f1_score.append(result.box.f1[0]) # Index 0 represnts "HD" (high damage)
  time_metrics.append((result.speed['preprocess'], result.speed['inference'], result.speed['postprocess']))

# Performance metrics
average_precision_damage = sum(precision_damage) / len(precision_damage)
average_recall_damage = sum(recall_damage) / len(recall_damage)
average_map_50 = sum(map_50) / len(map_50)
average_map_50_95 = sum(map_50_95) / len(map_50_95)
average_f1_score = sum(f1_score) / len(f1_score)

# Time metrics
num_time_metrics = len(time_metrics)
average_preprocessing_time = sum(metric[0] for metric in time_metrics) / num_time_metrics
average_inference_time = sum(metric[1] for metric in time_metrics) / num_time_metrics
average_postprocess_time = sum(metric[2] for metric in time_metrics) / num_time_metrics
average_total_time = average_preprocessing_time + average_inference_time + average_postprocess_time

# Print results from cross validation
print(f"Average precision: {average_precision_damage * 100:.2f}")
print(f"Average recall: {average_recall_damage * 100:.2f}")
print(f"Average mAP50: {average_map_50 * 100:.2f}")
print(f"Average mAP50-95: {average_map_50_95 * 100:.2f}")
print(f"Average f1: {average_f1_score * 100:.2f}")

print("\n")

print(f"Average preprocessing time: {average_preprocessing_time:.2f} ms")
print(f"Average inference time: {average_inference_time:.2f} ms")
print(f"Average postprocess_time: {average_postprocess_time:.2f} ms")
print(f"Average latency: {average_total_time:.2f} ms")

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.5, clip_limit=(4.0, 8.0), tile_grid_size=(16, 16))], auto_augment=randaugment, batch=10, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/YAMLS/folder_1.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01,

**Dowload results**

In [ ]:
import shutil
import os
import glob

# The data in /content/runs/detect/val, val-1, val-2 etc will be copied into here
saved_results = "/content/data/results"

os.makedirs(saved_results, exist_ok=True)

for folder in glob.glob("/content/runs/detect/val*"):
  shutil.copytree(folder, os.path.join(saved_results, os.path.basename(folder)))

shutil.make_archive("/content/data/saved_results" ,"zip", saved_results)


'/content/data/saved_results.zip'

In [ ]:
from google.colab import files
files.download("/content/data/saved_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>